In [1]:
# Merge samples into one file. 
library(tidyverse)
library(vroom)
library(data.table)

out_dir <- "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/"
dir.create(out_dir, showWarnings = FALSE)

# Define the process_samples function once
process_samples <- function(input_dir, sample_type, out_dir) {
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  
  input_filenames <- list.files(path = input_dir, pattern = "umi_dedup_normalized.tsv$", full.names = TRUE)
  
  # Precompute sample and condition for each file
  file_metadata <- tibble(
    filename = input_filenames,
    sample = basename(input_filenames) %>% str_extract(".+(?=_umi_dedup_normalized.tsv)"),
    condition = str_extract(basename(input_filenames), "^.+(?=-rep\\d)")
  )
  
  # Read data and attach metadata
  all_files_df <- map_dfr(seq_along(file_metadata$filename), function(i) {
    df <- vroom(file_metadata$filename[i], delim = ",")
    df$sample <- file_metadata$sample[i]
    df$condition <- file_metadata$condition[i]
    df
  })
  
  fwrite(all_files_df, file.path(out_dir, paste0(sample_type, "_all_samples_raw_counts.csv")))
}



# Process WT samples
input_output_mapping <- list(
  list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/WT", 
       sample_type = "WT", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/"),
   list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/MUT", 
       sample_type = "MUT", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/"),
   list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/OEx", 
       sample_type = "OEx", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/")       
)

# Process each sample type
walk(input_output_mapping, ~process_samples(.x$input_dir, .x$sample_type, .x$out_dir))


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘vroom’


The following objects are masked from ‘package:readr’:

    as.col_spec, col_character, col_date, col_datetime, col_double,
    col_factor, col_guess, col_integer, col_logical, col_number,
    col_skip, col_time, cols, cols_condense, cols_only, date_names,
    date_names_lang, date_names_langs, default_locale, fwf_cols,
    fwf_empty, fwf_positions, fwf_widths, locale, output_column,
    problems, spec



Attaching package: ‘data.table’


The

In [2]:
# Define the process_samples function once
process_individual_samples <- function(input_dir, sample_type, out_dir) {
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  
  input_filenames <- list.files(path = input_dir, pattern = "umi_dedup_normalized_tmp.tsv$", full.names = TRUE)
  
  # Precompute sample and condition for each file
  file_metadata <- tibble(
    filename = input_filenames,
    sample = basename(input_filenames) %>% 
      str_extract("(?<=-individual-).+(?=_umi_dedup_normalized_tmp.tsv$)"),
    condition = basename(input_filenames) %>%
      str_extract("(?<=-individual-).+?(?=-rep)")
  ) %>%
    mutate(
      condition = if_else(
        is.na(condition),
        basename(input_filenames) %>% str_extract("(?<=-individual-).+?(?=-\\S\\d+_S\\d+)"),
        condition
      )
    )

  
  # Read data and attach metadata
  all_files_df <- map_dfr(seq_along(file_metadata$filename), function(i) {
    df <- vroom(file_metadata$filename[i], delim = ",")
    # Remove the filename column
    df <- df %>% select(-filename)
    df$sample <- file_metadata$sample[i]
    df$condition <- file_metadata$condition[i]
    df
  })
  
  fwrite(all_files_df, file.path(out_dir, paste0(sample_type, "_all_samples_raw_counts_individual.csv")))
}

input_output_mapping_individual <- list(
  list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/WT_individual", 
       sample_type = "WT", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/"),
  list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/MUT_individual", 
       sample_type = "MUT", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/"),
  list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/OEx_individual", 
       sample_type = "OEx", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/")
)
walk(input_output_mapping_individual, ~process_individual_samples(.x$input_dir, .x$sample_type, .x$out_dir))

Rows: 178630 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): filename, index, mode, offset
dbl (3): insert_size, count, count_scaled

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 177292 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): filename, index, mode, offset
dbl (3): insert_size, count, count_scaled

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 155026 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): filename, index, mode, offset
dbl (3): insert_size, count, count_scaled

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the 